# Gold - Modelo Analítico, KPIs y Power BI

Objetivo de Gold: transformar los datasets curados de Silver en un modelo analítico listo para toma de decisiones y consumo en Power BI. Gold es la única capa que crea dimensiones, hechos, marts y KPIs.

Regla de arquitectura: Gold no lee CSV ni Raw. Todas las entradas analíticas vienen de `data/silver` en Parquet y las salidas se publican en `data/gold` también como Parquet Snappy.


In [ ]:
# Configura Spark y funciones de lectura para Silver y Gold en Parquet.
# Gold concentra el modelo dimensional que sera consumido por Power BI.
# Las funciones reducen codigo repetido al cargar tablas del lake.
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, types as T

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if PROJECT_ROOT.name in {"bronze", "silver", "gold"}:
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

spark = (
    SparkSession.builder
    .appName("municipal-medallion-profiling")
    .config("spark.sql.parquet.mergeSchema", "true")
    .getOrCreate()
)

def path_exists(path: str) -> bool:
    return Path(path).exists()

def read_parquet(path: str):
    if not path_exists(path):
        print(f"No existe: {path}")
        return None
    return spark.read.parquet(path)

def show_df(df, n=10, truncate=False):
    if df is None:
        print("DataFrame no disponible")
    else:
        df.show(n, truncate=truncate)

def count_nulls_and_blanks(df):
    exprs = []
    for c, dtype in df.dtypes:
        if dtype == "string":
            exprs.append(F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c))
        else:
            exprs.append(F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c))
    return df.select(exprs)

def summarize_table(name: str, df, business_keys=None):
    business_keys = business_keys or []
    rows = df.count()
    cols = len(df.columns)
    duplicates = rows - df.dropDuplicates().count()
    print(f"Tabla: {name}")
    print(f"Registros: {rows:,}")
    print(f"Columnas: {cols}")
    print(f"Duplicados exactos: {duplicates:,}")
    if business_keys and all(c in df.columns for c in business_keys):
        dup_keys = df.groupBy(*business_keys).count().filter("count > 1").count()
        print(f"Duplicados por clave {business_keys}: {dup_keys:,}")
    df.printSchema()
    return {"table": name, "rows": rows, "columns": cols, "duplicates": duplicates}

def domain_check(df, column, valid_values):
    return (
        df.groupBy(column)
        .count()
        .withColumn("is_valid_domain", F.col(column).isin(list(valid_values)))
        .orderBy(F.desc("count"))
    )


## 1. Lectura desde Silver Parquet

Objetivo: evidenciar que Gold parte de Silver, no de CSV. Este bloque inspecciona las tablas Silver que alimentan el modelo Gold.


In [ ]:
# Carga los datasets Silver que alimentan las transformaciones Gold.
# Gold no consume CSV ni raw directamente; parte de datos curados en Parquet.
# Esta separacion mantiene la arquitectura Medallion consistente.
silver_root = PROJECT_ROOT / "data" / "silver"
gold_root = PROJECT_ROOT / "data" / "gold"

silver_inputs = {
    "municipalidades_curated": read_parquet(str(silver_root / "municipalidades_curated")),
    "renamu_curated": read_parquet(str(silver_root / "renamu_curated")),
    "ingresos_municipales_curated": read_parquet(str(silver_root / "ingresos_municipales_curated")),
    "predial_esat_curated": read_parquet(str(silver_root / "predial_esat_curated")),
    "sismepre_entidad_estado_curated": read_parquet(str(silver_root / "sismepre_entidad_estado_curated")),
    "sismepre_respuestas_curated": read_parquet(str(silver_root / "sismepre_respuestas_curated")),
    "categorias_municipalidades_curated": read_parquet(str(silver_root / "categorias_municipalidades_curated")),
}

inventory = []
for name, df in silver_inputs.items():
    path = silver_root / name
    inventory.append((name, path.exists(), len(list(path.rglob("*.parquet"))) if path.exists() else 0, df.count() if df is not None else 0))
spark.createDataFrame(inventory, ["silver_dataset", "path_exists", "parquet_files", "rows"]).show(50, truncate=False)


## 2. Modelo dimensional Gold: dimensiones y facts

En Gold recien aparece el modelado dimensional. Silver entrega datasets limpios; Gold los convierte en tablas de analisis para Power BI.

Para exponer, muestra esta idea:

- **Dimensiones (`dim_*`)**: describen el contexto del analisis: municipalidad, tiempo, ubigeo, clasificador y catalogos SISMEPRE.
- **Facts (`fact_*`)**: contienen metricas medibles: ingresos, predial, cumplimiento, gestion tributaria, software y calidad.

Las tablas auxiliares para Power BI no reemplazan el modelo dimensional. El modelo principal se explica con dimensiones y facts.


In [ ]:
# Carga el inventario principal de dimensiones y hechos Gold.
# Las dimensiones describen entidades de analisis y las facts almacenan metricas.
# Esta estructura permite construir dashboards sin reprocesar Silver.
gold_tables = {
    "dim_municipalidad_gold": read_parquet(str(gold_root / "dim_municipalidad_gold")),
    "dim_ubigeo": read_parquet(str(gold_root / "dim_ubigeo")),
    "dim_tiempo": read_parquet(str(gold_root / "dim_tiempo")),
    "dim_clasificador_ingreso": read_parquet(str(gold_root / "dim_clasificador_ingreso")),
    "dim_estado_sismepre": read_parquet(str(gold_root / "dim_estado_sismepre")),
    "dim_formulario_sismepre": read_parquet(str(gold_root / "dim_formulario_sismepre")),
    "dim_pregunta_sismepre": read_parquet(str(gold_root / "dim_pregunta_sismepre")),
    "fact_ingresos_mensuales": read_parquet(str(gold_root / "fact_ingresos_mensuales")),
    "fact_ingresos_clasificador": read_parquet(str(gold_root / "fact_ingresos_clasificador")),
    "fact_predial_mensual": read_parquet(str(gold_root / "fact_predial_mensual")),
    "fact_sismepre_cumplimiento": read_parquet(str(gold_root / "fact_sismepre_cumplimiento")),
    "fact_sismepre_respuestas_resumen": read_parquet(str(gold_root / "fact_sismepre_respuestas_resumen")),
    "fact_renamu_gestion_tributaria": read_parquet(str(gold_root / "fact_renamu_gestion_tributaria")),
    "fact_renamu_software_at": read_parquet(str(gold_root / "fact_renamu_software_at")),
    "fact_calidad_datos": read_parquet(str(gold_root / "fact_calidad_datos")),
}

inventory_rows = []
for name, df in gold_tables.items():
    layer_type = "Dimension" if name.startswith("dim_") else "Fact"
    inventory_rows.append((name, layer_type, df.count(), len(df.columns)))

spark.createDataFrame(inventory_rows, ["tabla_gold", "tipo", "registros", "columnas"]).orderBy("tipo", "tabla_gold").show(50, truncate=False)


## 3. Relaciones del modelo

El modelo funciona como constelacion: varias tablas de hechos comparten dimensiones comunes.

Relaciones principales para explicar:

- `dim_municipalidad_gold[SEC_EJEC]` se conecta con facts SIAF, SISMEPRE y predial.
- `dim_municipalidad_gold[UBIGEO]` permite enriquecer con RENAMU y territorio.
- `dim_tiempo[periodo_id]` se conecta con hechos mensuales de ingresos.
- `dim_clasificador_ingreso[clasificador_id]` se conecta con el fact de ingresos por clasificador.
- Catalogos SISMEPRE explican estados, formularios y preguntas.


In [ ]:
# Define las relaciones logicas entre dimensiones y hechos.
# SEC_EJEC conecta municipalidades con ingresos, predial, RENAMU y SISMEPRE.
# La relacion temporal se apoya en campos year, ANO_DOC o ANO_ESTADISTICA.
relationship_summary = [
    ("dim_municipalidad_gold", "SEC_EJEC", "fact_ingresos_mensuales", "SEC_EJEC", "Municipalidad -> ingresos mensuales"),
    ("dim_municipalidad_gold", "SEC_EJEC", "fact_ingresos_clasificador", "SEC_EJEC", "Municipalidad -> ingresos por clasificador"),
    ("dim_municipalidad_gold", "SEC_EJEC", "fact_predial_mensual", "SEC_EJEC", "Municipalidad -> predial"),
    ("dim_municipalidad_gold", "SEC_EJEC", "fact_sismepre_cumplimiento", "SEC_EJEC", "Municipalidad -> cumplimiento SISMEPRE"),
    ("dim_tiempo", "periodo_id", "fact_ingresos_mensuales", "periodo_id", "Tiempo -> ingresos mensuales"),
    ("dim_clasificador_ingreso", "clasificador_id", "fact_ingresos_clasificador", "clasificador_id", "Clasificador -> ingresos"),
]

spark.createDataFrame(relationship_summary, ["dimension", "clave_dimension", "fact", "clave_fact", "uso"]).show(truncate=False)


## 4. KPIs de negocio desde facts y dimensiones

Estos KPIs se calculan desde tablas `fact_*`. No dependen de una tabla plana especial.


### KPI 1: Porcentaje de ejecucion de ingresos

**Que mide:** cuanto se recaudo frente al presupuesto modificado.

**Formula:** `MONTO_RECAUDADO / MONTO_PIM`.

**Relevancia:** permite saber si la municipalidad esta cumpliendo su capacidad de recaudacion presupuestada.


In [ ]:
# Calcula KPI de ejecucion de ingresos municipales.
# Agrega PIA, PIM y recaudado por municipalidad y calcula porcentaje de ejecucion.
# El calculo evita division entre cero cuando PIM no tiene valor.
income = gold_tables["fact_ingresos_mensuales"]
municipalities = gold_tables["dim_municipalidad_gold"]

kpi_ejecucion = (
    income.groupBy("SEC_EJEC")
    .agg(
        F.sum("MONTO_PIM").alias("TOTAL_PIM"),
        F.sum("MONTO_RECAUDADO").alias("TOTAL_RECAUDADO"),
    )
    .withColumn("INDICE_EJECUCION_PORC", F.when(F.col("TOTAL_PIM") != 0, F.col("TOTAL_RECAUDADO") / F.col("TOTAL_PIM") * 100))
    .join(municipalities.select("SEC_EJEC", "MUNICIPALIDAD_NOMBRE", "DEPARTAMENTO_NOMBRE", "categoria_municipalidad"), "SEC_EJEC", "left")
)

kpi_ejecucion.orderBy(F.desc("TOTAL_RECAUDADO")).show(10, truncate=False)


### KPI 2: Brecha de recaudacion

**Que mide:** diferencia entre presupuesto modificado y recaudacion real.

**Formula:** `MONTO_PIM - MONTO_RECAUDADO`.

**Relevancia:** ayuda a priorizar municipalidades donde falta recuperar ingresos.


In [ ]:
# Calcula brecha de recaudacion municipal.
# La brecha se obtiene restando recaudado al presupuesto modificado PIM.
# Este indicador ayuda a priorizar municipalidades con mayor diferencia pendiente.
kpi_brecha = (
    kpi_ejecucion
    .withColumn("BRECHA_RECAUDACION", F.col("TOTAL_PIM") - F.col("TOTAL_RECAUDADO"))
    .select("SEC_EJEC", "MUNICIPALIDAD_NOMBRE", "DEPARTAMENTO_NOMBRE", "categoria_municipalidad", "TOTAL_PIM", "TOTAL_RECAUDADO", "BRECHA_RECAUDACION")
)

kpi_brecha.orderBy(F.desc("BRECHA_RECAUDACION")).show(10, truncate=False)


### KPI 3: Efectividad predial

**Que mide:** que porcentaje de la emision predial fue recaudado.

**Formula:** `MON_RECAUDACION_TOTAL / MON_EMISIONPREDIAL_AFECTO`.

**Relevancia:** permite evaluar desempeno en impuesto predial, uno de los ingresos municipales mas importantes.


In [ ]:
# Calcula indicadores prediales agregados.
# Se resume recaudacion, emision y efectividad predial por municipalidad.
# El indicador permite comparar desempeno tributario predial entre territorios.
predial = gold_tables["fact_predial_mensual"]

kpi_predial = (
    predial.groupBy("SEC_EJEC")
    .agg(
        F.sum("MON_RECAUDACION_TOTAL").alias("RECAUDACION_PREDIAL"),
        F.sum("MON_EMISIONPREDIAL_AFECTO").alias("EMISION_PREDIAL"),
        F.sum("MON_SALDO_PREDIAL_TOTAL").alias("SALDO_PREDIAL"),
    )
    .withColumn("EFECTIVIDAD_PREDIAL_PORC", F.when(F.col("EMISION_PREDIAL") != 0, F.col("RECAUDACION_PREDIAL") / F.col("EMISION_PREDIAL") * 100))
    .join(municipalities.select("SEC_EJEC", "MUNICIPALIDAD_NOMBRE", "DEPARTAMENTO_NOMBRE", "categoria_municipalidad"), "SEC_EJEC", "left")
)

kpi_predial.orderBy(F.desc("SALDO_PREDIAL")).show(10, truncate=False)


### KPI 4: Capacidad tecnologica tributaria

**Que mide:** si la municipalidad usa SRTM, software de rentas, catastro o al menos una herramienta tributaria.

**Relevancia:** conecta gestion administrativa RENAMU con desempeno de recaudacion.


In [ ]:
# Calcula capacidad tecnologica tributaria con datos RENAMU.
# Se combinan indicadores de SRTM, software de rentas y catastro.
# La metrica permite analizar si la tecnologia acompana mejores resultados.
software = gold_tables["fact_renamu_software_at"]

enriched_software = (
    software.join(municipalities.select("SEC_EJEC", "MUNICIPALIDAD_NOMBRE", "DEPARTAMENTO_NOMBRE", "categoria_municipalidad"), "SEC_EJEC", "left")
)

kpi_software = (
    enriched_software.groupBy("DEPARTAMENTO_NOMBRE", "categoria_municipalidad")
    .agg(
        F.countDistinct("SEC_EJEC").alias("MUNICIPALIDADES"),
        F.sum(F.col("usa_srtm_estado").cast("int")).alias("CON_SRTM"),
        F.sum(F.col("usa_software_rentas_at").cast("int")).alias("CON_SOFTWARE_RENTAS"),
        F.sum(F.col("usa_software_catastro").cast("int")).alias("CON_CATASTRO"),
        F.sum(F.col("usa_al_menos_un_software_at").cast("int")).alias("CON_AL_MENOS_UN_SOFTWARE"),
    )
)

kpi_software.orderBy("DEPARTAMENTO_NOMBRE", "categoria_municipalidad").show(20, truncate=False)


## 5. Validaciones de calidad Gold

Aqui se valida que el modelo dimensional no pierda datos y conserve claves consistentes.


In [ ]:
# Ejecuta validaciones basicas del modelo Gold.
# Se revisa unicidad de dimensiones, integridad de claves y consistencia de hechos.
# Estas pruebas protegen la confiabilidad de los KPIs.
validations = []

# La dimension municipal debe tener una sola fila por SEC_EJEC.
dim_muni = gold_tables["dim_municipalidad_gold"]
dim_duplicates = dim_muni.groupBy("SEC_EJEC").count().filter("count > 1").count()
validations.append(("dim_municipalidad_gold", "SEC_EJEC unico", "passed" if dim_duplicates == 0 else "failed", dim_duplicates))

# Las facts principales no deben tener claves municipales fuera de la dimension.
for fact_name in ["fact_ingresos_mensuales", "fact_ingresos_clasificador", "fact_predial_mensual", "fact_sismepre_cumplimiento"]:
    fact = gold_tables[fact_name]
    missing = fact.select("SEC_EJEC").dropDuplicates().join(dim_muni.select("SEC_EJEC"), "SEC_EJEC", "left_anti").count()
    validations.append((fact_name, "Integridad referencial SEC_EJEC", "passed" if missing == 0 else "failed", missing))

# La categoria debe estar dentro del dominio A-G cuando existe.
invalid_categories = dim_muni.filter("categoria_municipalidad IS NOT NULL AND categoria_municipalidad NOT IN ('A','B','C','D','E','F','G')").count()
validations.append(("dim_municipalidad_gold", "Dominio categoria A-G", "passed" if invalid_categories == 0 else "failed", invalid_categories))

spark.createDataFrame(validations, ["tabla", "validacion", "estado", "detalle"]).show(truncate=False)


## 6. Tablas para construir los seis dashboards

Los dashboards se construyen desde dimensiones y facts. El modelo dimensional es la fuente principal; cualquier salida auxiliar para consumo rapido no reemplaza el modelo correcto.


In [ ]:
# Mapea cada dashboard a las tablas dimensionales que necesita.
# La lista sirve como guia para armar Power BI con dim_* y fact_*.
# No depende de tablas auxiliares planas para mantener el modelo reutilizable.
dashboard_sources = [
    ("Dashboard 1", "Recaudacion municipal vs capacidad tributaria", "dim_municipalidad_gold, dim_tiempo, fact_ingresos_mensuales, fact_renamu_gestion_tributaria"),
    ("Dashboard 2", "Recaudacion por clasificador de ingreso", "dim_municipalidad_gold, dim_tiempo, dim_clasificador_ingreso, fact_ingresos_clasificador"),
    ("Dashboard 3", "Predial vs efectividad", "dim_municipalidad_gold, fact_predial_mensual, fact_sismepre_cumplimiento"),
    ("Dashboard 4", "Distribucion de efectividad predial", "dim_municipalidad_gold, fact_predial_mensual"),
    ("Dashboard 5", "Software tributario municipal", "dim_municipalidad_gold, fact_renamu_software_at, fact_renamu_gestion_tributaria"),
    ("Dashboard 6", "Priorizacion municipal", "dim_municipalidad_gold, fact_ingresos_mensuales, fact_predial_mensual, fact_renamu_software_at, fact_sismepre_cumplimiento"),
]

spark.createDataFrame(dashboard_sources, ["dashboard", "objetivo", "tablas_dim_fact"]).show(truncate=False)


## 7. Ejemplos visuales rapidos

Estas consultas muestran que las dimensiones y facts ya permiten generar indicadores para Power BI.


In [ ]:
# Prepara agregacion de recaudacion por departamento y categoria.
# Combina fact_ingresos_mensuales con dim_municipalidad_gold.
# Esta salida se usa para rankings territoriales y segmentadores A-G.
recaudacion_departamento = (
    income.join(municipalities.select("SEC_EJEC", "DEPARTAMENTO_NOMBRE", "categoria_municipalidad"), "SEC_EJEC", "left")
    .groupBy("DEPARTAMENTO_NOMBRE", "categoria_municipalidad")
    .agg(F.sum("MONTO_RECAUDADO").alias("TOTAL_RECAUDADO"))
    .orderBy(F.desc("TOTAL_RECAUDADO"))
)
recaudacion_departamento.show(20, truncate=False)


In [ ]:
# Prepara ranking de recaudacion por clasificador de ingreso.
# Cruza la fact por clasificador con su dimension descriptiva.
# Permite analizar rubros y especificas con mayor aporte a la recaudacion.
classifier = gold_tables["dim_clasificador_ingreso"]
fact_classifier = gold_tables["fact_ingresos_clasificador"]

ranking_clasificador = (
    fact_classifier.join(classifier.select("clasificador_id", "ESPECIFICA_DET_NOMBRE", "RUBRO_NOMBRE"), "clasificador_id", "left")
    .groupBy("RUBRO_NOMBRE", "ESPECIFICA_DET_NOMBRE")
    .agg(F.sum("MONTO_RECAUDADO").alias("TOTAL_RECAUDADO"))
    .orderBy(F.desc("TOTAL_RECAUDADO"))
)
ranking_clasificador.show(20, truncate=False)


## 8. Persistencia final en Parquet

Gold queda publicado en `data/gold` como carpetas Parquet. Para Power BI se pueden cargar directamente las dimensiones y facts necesarias.


In [ ]:
# Verifica persistencia fisica de dimensiones y hechos Gold.
# Cada tabla debe existir como carpeta Parquet con archivos part-*.parquet.
# Esta revision confirma que las salidas estan listas para consumo analitico.
for table in gold_tables:
    table_path = gold_root / table
    parquet_files = list(table_path.rglob("*.parquet")) if table_path.exists() else []
    print(f"{table}: {len(parquet_files)} archivos parquet")


## 9. Conclusion Gold

Gold consume Silver Parquet y construye un modelo dimensional de tipo constelacion:

- Dimensiones para describir municipalidad, tiempo, territorio, clasificador y catalogos.
- Facts para medir ingresos, predial, cumplimiento, software, gestion tributaria y calidad.
- KPIs calculados desde facts, listos para Power BI.

Esta es la parte que debes defender como modelo analitico final.
